# Dipole Angle vs Parallactic Angle Correlation

This notebook demonstrates that the **raw dipole angle** (`r:dipoleAngle`) measured by the
Rubin/LSST AP pipeline is fully correlated with the **parallactic angle** η computed from
first principles (RA, Dec, MJD → hour angle → arctan2 formula).

Both quantities peak at **−90° and +90°** relative to the North–South direction, which
confirms that the dipoles are driven by atmospheric differential refraction / PSF elongation
along the zenith direction.

## Angle conventions used

| Column | Convention | Range |
|--------|------------|-------|
| `r:dipoleAngle` | CCW from East (+x pixel axis) | −180 … +180° or 0 … 360° |
| `parallactic_angle_deg` | CCW from North (astronomical PA) | −180 … +180° |
| `azimuth_deg` | CW from North (astropy standard) | 0 … 360° |
| `zenith_angle_deg` | 90° − altitude | 0 … 90° |
| `sin_zenith` | sin(zenith_angle_deg) | 0 … 1 |
| `airmass` | ≈ 1/cos(z) | ≥ 1 |
| `hour_angle_hr` | H = LST − RA, wrapped to (−12h, +12h] | ±12 h |
| `hour_angle_deg` | same, in degrees | ±180° |

**No dipole_PA conversion is needed** — we work directly with `r:dipoleAngle`
and the parallactic angle in their native ranges.

## Physical picture

Atmospheric differential chromatic refraction (DCR) displaces each source along the
great circle toward the zenith.  In the tangent plane this direction is exactly the
parallactic angle η.  Because the AP pipeline subtracts a template taken at a
**different** airmass (and possibly a different hour angle), the residual PSF elongation
generates a dipole whose axis tracks η.  The **amplitude** of DCR scales as
tan z ≈ sin z for moderate zenith angles, so `r:dipoleLength` should grow with sin z.

The algorithm may orient the positive lobe of the dipole in either direction along η
(the sign is not physically meaningful), hence:
* the signed difference `delta_dipole_para = r:dipoleAngle − η` clusters near both 0° and ±180°;
* the headless (folded) difference `delta_dipole_para_folded` is the minimum of
  |Δ| and 180° − |Δ| and clusters near 0°.

## Strategy

* Load dipole alerts from `data_DIPOLES_01c/` parquet files.
* Compute η, H, azimuth, zenith, sin z and airmass for every alert with `astropy`.
* 360° rose diagrams stacked by band, 2×3 DDF grid: `r:dipoleAngle`, azimuth, η, Δ signed, Δ folded.
* Bar-plot distributions of zenith and airmass (stacked by band, 2×3 DDF).
* `r:dipoleLength` vs sin z — scatter + median profile ± MAD (2×3 DDF + all-DDFs panel).
* `r:dipoleLength` vs |H| and `r:dipoleAngle` vs H — scatter + median profile (2×3 DDF + combined).
* Pearson/Spearman correlation table and Spearman ρ heatmaps.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-28
- last update : 2026-05-30 : add sin(z) and hour-angle analyses, Δ signed + folded rose diagrams

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from scipy import stats

from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u
from astropy.timeseries import TimeSeries
from astropy.coordinates import get_sun

warnings.filterwarnings("ignore")
print(f"pandas  {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
# astroplan to check
from astroplan import Observer
from astroplan import FixedTarget
from astroplan.plots import plot_airmass, plot_parallactic, plot_altitude, plot_sky
from astroplan import is_observable

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("ipympl not found → %matplotlib inline")

In [ ]:
# ── I/O paths ─────────────────────────────────────────────────────────────────
DIR_DATA_IN = "data_DIPOLES_01c"
NB_TAG = "DIPOLES_06"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figures: {os.path.abspath(DIR_FIGS)}")

# ── Rubin/LSST – Cerro Pachón ─────────────────────────────────────────────────
RUBIN_LAT_DEG = -30.244728
RUBIN_LON_DEG = -70.749417
RUBIN_HEIGHT_M = 2647.0
RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Observatory: lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

# --- Astroplan observer ----------------------------------------------
observer = Observer.at_site("lsst", timezone="UTC")

# ── LSST Deep Drilling Fields ─────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}
DDF_NAMES = list(DEEP_FIELDS.keys())  # fixed order for 2×3 grid


DEEP_FIELDS_COLORSTYLE = {
    "COSMOS": {"color": "r"},
    "ELAIS-S1": {"color": "k"},
    "ECDFS": {"color": "grey"},
    "EDFS-a": {"color": "b"},
    "EDFS-b": {"color": "g"},
    "EDFS": {"color": "magenta"},
    "M49": {"color": "purple"},
}


# ── Band colours (LSST ugrizy) ────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

# ── Matplotlib defaults ───────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure as PDF + PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Observing-geometry helper

The parallactic angle and the hour angle are both computed from the LST:

$$H = \mathrm{LST} - \alpha \qquad\qquad
\eta = \arctan2\!\left(\sin H,\;\tan\phi\cos\delta - \sin\delta\cos H\right)$$

The function returns η, H (hours and degrees), azimuth, zenith angle, sin z, and airmass.

### 2a. `zenith_tangent_vector` — tangent-plane projection (from obstime)

Projects the zenith direction into the tangent plane of the target using
the formula $\mathbf{v} = \mathbf{z} - (\mathbf{z}\cdot\mathbf{s})\,\mathbf{s}$,
where $\mathbf{s}$ is the unit vector toward the source and $\mathbf{z}$ is the
zenith unit vector obtained by transforming AltAz(alt=90°) to ICRS.
This version calls `astropy` for the time transform and is used for individual
sanity checks.

In [ ]:
def zenith_tangent_vector(ra_deg, dec_deg, obstime, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith

    Parameters:
    ==========
        ra_deg,dec_deg: target coordinates in the sky
        obstimes:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
    """

    # Source
    sky = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg)

    # Zénith en AltAz → (alt=90°, az arbitraire)
    zenith_altaz = SkyCoord(alt=90 * u.deg, az=0 * u.deg, frame=AltAz(obstime=obstime, location=location))

    # Convertir en ICRS
    zenith_icrs = zenith_altaz.transform_to("icrs")

    # Vecteurs cartésiens
    s = sky.cartesian.xyz.value
    z = zenith_icrs.cartesian.xyz.value

    # Projection tangentielle
    v = z - np.dot(z, s) * s

    # Normalisation
    v /= np.linalg.norm(v)

    return v  # vecteur 3D tangent au ciel

### 2b. `zenith_tangent_vector_fromHA` — tangent-plane projection (from HA grid)

Same projection as above but computed analytically from a **precomputed hour-angle array** —
avoids repeated `astropy` time calls and is fast enough to sweep the full
$H\in[-180°,+180°]$ range for all DDFs.
The function also returns $\|\mathbf{v}\| = \sin z$, the zenith-angle sine
that governs DCR amplitude.

In [ ]:
def zenith_tangent_vector_fromHA(HA_deg, coords, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith
    Parameters:
    ==========
        HA_deg : array of Hour angles
        coords: target SkyCoords
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
        array if sinz values (related to dipole intensity)
    """

    lat_deg = location.lat.to(u.deg).value

    dec_deg = coords.dec.to(u.deg).value
    ra_deg = coords.ra.to(u.deg).value

    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)

    # --- direction source ---
    s = np.array([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])  # (3,)

    # --- zénith ---
    HA_val = HA_deg.to(u.deg).value  # ← FIX unités
    lst = np.deg2rad(HA_val + ra_deg)
    lat = np.deg2rad(lat_deg)

    z = np.array(
        [np.cos(lat) * np.cos(lst), np.cos(lat) * np.sin(lst), np.sin(lat) * np.ones_like(lst)]
    )  # (3, N)

    # --- projection ---
    # v = z - np.dot(z, s) * s
    proj = np.sum(z * s[:, None], axis=0)  # (N,)
    v = z - proj * s[:, None]  # (3, N)

    # --- norme par point ---
    norm = np.linalg.norm(v, axis=0)  # (N,)

    # --- normalisation optionnelle ---
    v_unit = np.zeros_like(v)
    mask = norm > 0
    v_unit[:, mask] = v[:, mask] / norm[mask]

    return v_unit, norm

### 2c. `sinz_vs_HA` — zenith-angle sine from analytic formula

Direct analytic computation:
$$\sin z(H,\delta,\phi) = \sqrt{1 - \left(\sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H\right)^2}$$
Used to cross-check the norm returned by `zenith_tangent_vector_fromHA`
and to overlay the alert's measured $\sin z$ on the model curve.

In [ ]:
def sinz_vs_HA(HA_deg, coords, location):
    """
    Compute the sinus of zenith angle from the formula
    \sin z(H,\delta,\phi) = \sqrt{ 1 - \left( \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H \right)^2

    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree

    """

    # --- location ---> latitude
    lat_deg = location.lat.to(u.deg).value

    # --- object ---> declination
    dec_deg = coords.dec.to(u.deg).value

    # --- HA be sure to have quanitites in deg
    HA_valdeg = HA_deg.to(u.deg).value

    HA = np.deg2rad(HA_valdeg)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)

    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    return np.sqrt(1 - cosz**2)

### 2d. `calculate_parallactic_angle` — η from obstime

Computes the parallactic angle from first principles given `astropy` `Time`
objects:
$$H = \mathrm{LST} - \alpha, \qquad
\eta = \arctan2\!\left(\sin H,\;\tan\phi\,\cos\delta - \sin\delta\,\cos H\right)$$
Returns η in degrees, range $[-180°, +180°]$.

In [ ]:
# -----------------------------
# My computation of  parallactic angle
# -----------------------------
def calculate_parallactic_angle(coords, times, location):
    """
    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree
    """

    # LST
    lst = times.sidereal_time("apparent", longitude=location.lon)

    # angle horaire H = LST - RA
    H = (lst - coords.ra).to(u.rad).value

    # latitude
    phi = location.lat.to(u.rad).value

    # déclinaison
    dec_rad = coords.dec.to(u.rad).value

    # formule du parallactic angle
    sinH = np.sin(H)
    cosH = np.cos(H)

    tan_phi = np.tan(phi)

    num = sinH
    den = tan_phi * np.cos(dec_rad) - np.sin(dec_rad) * cosH

    q = np.arctan2(num, den)

    return np.degrees(q)

### 2e. `calculate_parallactic_angle_fromHA` — η from HA grid

Same formula as above but accepts a **precomputed hour-angle** `Angle` object
(in degrees) instead of `Time`.  Used to sweep the full HA range for
all DDFs without triggering IERS/UT1 network calls.

In [ ]:
# -----------------------------
# My computation of  parallactic angle
# -----------------------------
def calculate_parallactic_angle_fromHA(ha, coords, location):
    """
    Parameters:
    ==========
        coords: target SkyCoords
        ha:  hour angle Angle in degree
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree
    """

    # angle horaire H = LST - RA
    ha_rad = ha.to(u.rad).value

    # latitude
    phi = location.lat.to(u.rad).value

    # déclinaison
    dec_rad = coords.dec.to(u.rad).value

    # formule du parallactic angle
    sinH = np.sin(ha_rad)
    cosH = np.cos(ha_rad)

    tan_phi = np.tan(phi)

    num = sinH
    den = tan_phi * np.cos(dec_rad) - np.sin(dec_rad) * cosH

    q = np.arctan2(num, den)

    return np.degrees(q)

### 2f. `compute_observing_geometry` — full geometry pipeline

Main workhorse: given arrays of (RA, Dec, MJD), returns a `DataFrame` with
all derived observing-geometry columns (η, H, azimuth, altitude, zenith,
$\sin z$, airmass).  Processes alerts in batches of `batch_size` to keep
memory usage bounded.  A quick sanity check on COSMOS is run at the end.

In [ ]:
def compute_observing_geometry(
    ra_deg: np.ndarray,
    dec_deg: np.ndarray,
    mjd: np.ndarray,
    location: EarthLocation = RUBIN_LOCATION,
    batch_size: int = 500,
) -> pd.DataFrame:
    """
    Compute full observing geometry for a set of alerts.

    Parameters
    ----------
    ra_deg, dec_deg : array-like – ICRS coordinates in degrees
    mjd             : array-like – MJD TAI
    location        : EarthLocation
    batch_size      : int – alerts per astropy call (speed/memory trade-off)

    Returns
    -------
    pd.DataFrame with columns:
        parallactic_angle_deg  float   −180 … +180°   (North = 0, CCW)
        hour_angle_hr          float   −12 … +12 h     (H = LST − RA)
        hour_angle_deg         float   −180 … +180°    (same × 15)
        azimuth_deg            float      0 … 360°    (North = 0, E = 90)
        altitude_deg           float      0 …  90°
        zenith_angle_deg       float      0 …  90°
        sin_zenith             float      0 … 1
        airmass                float   ≥ 1             (≈ 1/cos z)
    """
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    t = np.asarray(mjd, dtype=float)
    n = len(ra)

    para = np.full(n, np.nan)
    H_hr = np.full(n, np.nan)  # hour angle in hours
    az = np.full(n, np.nan)
    alt = np.full(n, np.nan)
    za = np.full(n, np.nan)

    for i0 in range(0, n, batch_size):
        sl = slice(i0, min(i0 + batch_size, n))
        try:
            # Sidereal time requires UT1
            times = Time(t[sl], format="mjd", scale="tai").ut1
            coords = SkyCoord(ra=ra[sl] * u.deg, dec=dec[sl] * u.deg)

            # LST and hour angle
            lst = times.sidereal_time("apparent", longitude=location.lon)
            H_wrap = (lst - coords.ra).wrap_at(180 * u.deg)  # Angle in (−180°, +180°]
            H_rad = H_wrap.to(u.rad).value
            H_hr[sl] = H_wrap.to(u.hourangle).value  # hours

            # Parallactic angle: η = arctan2(sin H, tan φ cos δ − sin δ cos H)
            phi = location.lat.to(u.rad).value
            dec_rad = coords.dec.to(u.rad).value
            para[sl] = np.degrees(
                np.arctan2(
                    np.sin(H_rad),
                    np.tan(phi) * np.cos(dec_rad) - np.sin(dec_rad) * np.cos(H_rad),
                )
            )

            # Alt/Az
            frame = AltAz(obstime=times, location=location)
            altaz = coords.transform_to(frame)
            alt[sl] = altaz.alt.deg
            az[sl] = altaz.az.deg
            za[sl] = 90.0 - altaz.alt.deg
        except Exception as exc:
            print(f"  [warning] batch {i0}–{i0 + batch_size}: {exc}")

    with np.errstate(divide="ignore", invalid="ignore"):
        airmass = np.where(za < 89.0, 1.0 / np.cos(np.radians(za)), np.nan)

    return pd.DataFrame(
        {
            "parallactic_angle_deg": para,
            "hour_angle_hr": H_hr,
            "hour_angle_deg": H_hr * 15.0,  # 1 h = 15°
            "azimuth_deg": az,
            "altitude_deg": alt,
            "zenith_angle_deg": za,
            "sin_zenith": np.sin(np.radians(za)),
            "airmass": airmass,
        }
    )


# Sanity check
test = compute_observing_geometry([150.1191], [2.2058], [60310.5])
print("Sanity check COSMOS MJD=60310.5:")
print(test.to_string(index=False))

## 3. Load dipole alerts

Dipole alerts have already been classified and saved to parquet by notebook
`01_fink_block_flatlightcurves.ipynb` (pipeline v6).  We load only the
`r:isDipole == True` subset and cast numeric columns.

In [ ]:
ddf_alerts: dict[str, pd.DataFrame] = {}

for field_name in DDF_NAMES:
    pq = os.path.join(DIR_DATA_IN, f"{field_name}_alerts.parquet")
    if not os.path.exists(pq):
        print(f"[{field_name:12s}] parquet not found — skipping.")
        ddf_alerts[field_name] = pd.DataFrame()
        continue

    df = pd.read_parquet(pq)

    # Cast boolean isDipole
    if "r:isDipole" in df.columns:
        df["r:isDipole"] = (
            df["r:isDipole"]
            .map(
                lambda v: (
                    True
                    if str(v).strip().lower() in ("true", "1", "yes")
                    else False
                    if str(v).strip().lower() in ("false", "0", "no")
                    else pd.NA
                )
            )
            .astype("boolean")
        )

    for col in (
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:dipoleAngle",
        "r:dipoleLength",
        "r:dipoleChi2",
        "r:dipoleFluxDiff",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df_dip = (
        df[df["r:isDipole"].fillna(False).astype(bool)].copy()
        if "r:isDipole" in df.columns
        else pd.DataFrame()
    )
    df_dip["field"] = field_name
    ddf_alerts[field_name] = df_dip
    print(f"[{field_name:12s}] {len(df):7,} total  |  {len(df_dip):6,} dipoles")

print("\nLoad complete.")

## 4. Compute observing geometry + derived quantities

New derived columns added here:

| Column | Definition |
|--------|------------|
| `sin_zenith` | sin(zenith_angle_deg) — proxy for DCR amplitude |
| `hour_angle_hr` | H = LST − RA in hours, wrapped to (−12h, +12h] |
| `hour_angle_deg` | same × 15, in degrees |
| `delta_dipole_para` | `r:dipoleAngle` − η, wrapped to (−180°, +180°] — signed difference |
| `delta_dipole_para_folded` | min(|Δ|, 180°−|Δ|) ∈ [0°, 90°] — headless (axis without direction) |

Calls `compute_observing_geometry` on every DDF subset, then derives the
signed and folded angular differences between `r:dipoleAngle` and η:

* **`delta_dipole_para`** $= r\!:\!\text{dipoleAngle} - \eta$, wrapped to
  $(-180°, +180°]$.
* **`delta_dipole_para_folded`** $= \min(|\Delta|,\,180°-|\Delta|)$, which
  treats the dipole axis as headless and maps both $0°$ and $180°$ to zero
  alignment.

In [ ]:
frames: list[pd.DataFrame] = []

for field_name in DDF_NAMES:
    df_dip = ddf_alerts.get(field_name, pd.DataFrame())
    if df_dip.empty:
        print(f"[{field_name:12s}] no dipoles — skipping.")
        continue

    need = ["r:ra", "r:dec", "r:midpointMjdTai"]
    missing = [c for c in need if c not in df_dip.columns]
    if missing:
        print(f"[{field_name:12s}] missing {missing} — skipping.")
        continue

    mask = df_dip["r:ra"].notna() & df_dip["r:dec"].notna() & df_dip["r:midpointMjdTai"].notna()
    df_clean = df_dip[mask].copy().reset_index(drop=True)
    print(f"[{field_name:12s}] computing geometry for {len(df_clean):,} dipoles …", end=" ")

    geo = compute_observing_geometry(
        ra_deg=df_clean["r:ra"].values,
        dec_deg=df_clean["r:dec"].values,
        mjd=df_clean["r:midpointMjdTai"].values,
    )
    df_clean = pd.concat([df_clean, geo], axis=1)

    # ── Angular differences between dipole angle and parallactic angle ──────
    if "r:dipoleAngle" in df_clean.columns:
        raw = df_clean["r:dipoleAngle"].values
        parang = df_clean["parallactic_angle_deg"].values

        # Signed difference wrapped to (−180°, +180°]
        diff = (raw - parang + 180.0) % 360.0 - 180.0
        df_clean["delta_dipole_para"] = diff

        # Headless (folded) difference: dipole axis has no preferred direction,
        # so 0° and 180° are equivalent.  Fold into [0°, 90°].
        # |Δ| mod 180 then fold to [0,90]
        adiff = np.abs(diff)  # [0, 180]
        folded = np.where(adiff <= 90.0, adiff, 180.0 - adiff)  # [0, 90]
        df_clean["delta_dipole_para_folded"] = folded

    frames.append(df_clean)
    print("done")

if frames:
    df_all = pd.concat(frames, ignore_index=True)
    print(f"\nTotal dipoles with geometry: {len(df_all):,}")
    cols_show = [
        "field",
        "r:band",
        "r:midpointMjdTai",
        "r:dipoleAngle",
        "r:dipoleLength",
        "parallactic_angle_deg",
        "hour_angle_hr",
        "hour_angle_deg",
        "zenith_angle_deg",
        "sin_zenith",
        "azimuth_deg",
        "altitude_deg",
        "airmass",
        "delta_dipole_para",
        "delta_dipole_para_folded",
    ]
    display(df_all[[c for c in cols_show if c in df_all.columns]].describe())
else:
    df_all = pd.DataFrame()
    print("No dipoles found — nothing to analyse.")

## 5. Analytical verification — η(H) and sin z(H) for all DDFs

Before checking individual alerts we verify that the two independent
implementations (`calculate_parallactic_angle_fromHA` and
`zenith_tangent_vector_fromHA` / `sinz_vs_HA`) produce identical curves
when swept over the full hour-angle range $H \in [-180°, +180°]$.

### 5a. Parallactic angle η vs hour angle H

### 5a. Parallactic angle η vs hour angle H

Sweeps $H\in[-180°,+180°]$ for all six DDFs using `calculate_parallactic_angle_fromHA`.
Curves are coloured by field; the $\delta$ value is shown in the legend.
At meridian transit ($H=0$) the parallactic angle is 0° for fields north of the
zenith and ±180° for fields south of the zenith at Cerro Pachón ($\phi \approx -30.2°$).

In [ ]:
figname = "ddf_parallacticvsha"

fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")

HA = np.arange(-180.0, 180.0) * u.deg

for key, value in DEEP_FIELDS.items():
    field_name = key
    ra = value[0]
    dec = value[1]
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    label = field_name + " : ( $\delta = $" + f"{dec:.1f} deg )"

    q = calculate_parallactic_angle_fromHA(HA, coords, RUBIN_LOCATION)

    ax.plot(HA, q, label=label, lw=2)

ax.set_xlabel("Hour angle (degrees)")
ax.set_ylabel("Parallactic angle (degrees)")


# --- Axe secondaire en heures ---
def deg2hour(x):
    return x / 15.0


def hour2deg(x):
    return x * 15.0


secax = ax.secondary_xaxis("top", functions=(deg2hour, hour2deg))
secax.set_xlabel("Hour angle (hours)")

ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"LSST Deep Fields parallactic angle vs hour angle")

savefig(figname)
plt.show()

### 5b. $\sin z$ via tangent-plane norm (`zenith_tangent_vector_fromHA`)

The norm $\|\mathbf{v}\|$ of the zenith projection vector equals $\sin z$.
Plotted here for all DDFs to confirm the analytic dependence on declination.

In [ ]:
figname = "ddf_sinzenithvsha"

fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")

HA = np.arange(-180.0, 180.0) * u.deg

for key, value in DEEP_FIELDS.items():
    field_name = key
    ra = value[0]
    dec = value[1]
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    label = field_name + " : ( $\delta = $" + f"{dec:.1f} deg )"

    # projection of zenith in object tangeant plane
    v, sinz = zenith_tangent_vector_fromHA(HA, coords, RUBIN_LOCATION)

    ax.plot(HA, sinz, label=label, lw=2)

ax.set_xlabel("Hour angle (degrees)")
ax.set_ylabel("sinz")


# --- Axe secondaire en heures ---
def deg2hour(x):
    return x / 15.0


def hour2deg(x):
    return x * 15.0


secax = ax.secondary_xaxis("top", functions=(deg2hour, hour2deg))
secax.set_xlabel("Hour angle (hours)")

ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"LSST Deep Fields sinz vs hour angle")

savefig(figname)

plt.show()

### 5c. $\sin z$ via direct formula (`sinz_vs_HA`) — cross-check

Cross-check using the closed-form expression.  Both panels (5b and 5c)
must be identical; any discrepancy would signal a sign or frame convention error.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")

HA = np.arange(-180.0, 180.0) * u.deg

for key, value in DEEP_FIELDS.items():
    field_name = key
    ra = value[0]
    dec = value[1]
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    label = field_name + " : ( $\delta = $" + f"{dec:.1f} deg )"

    # projection of zenith in object tangeant plane
    sinz = sinz_vs_HA(HA, coords, RUBIN_LOCATION)

    ax.plot(HA, sinz, label=label, lw=2)

ax.set_xlabel("Hour angle (degrees)")
ax.set_ylabel("sinz")


# --- Axe secondaire en heures ---
def deg2hour(x):
    return x / 15.0


def hour2deg(x):
    return x * 15.0


secax = ax.secondary_xaxis("top", functions=(deg2hour, hour2deg))
secax.set_xlabel("Hour angle (hours)")

ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"LSST Deep Fields sinz vs hour angle")

plt.show()

## 6. Single-alert numerical check

We pick one representative dipole alert (`idx_frame`, `idx_alert`) and
cross-validate **every** computed angle and observing-geometry quantity
against three independent references:

1. **My analytic model** — `calculate_parallactic_angle_fromHA`, `sinz_vs_HA`.
2. **`compute_observing_geometry`** — astropy-based pipeline used on the whole dataset.
3. **`astroplan`** — `plot_parallactic`, `plot_airmass`, `plot_sky`.

All three should give the same η and $\sin z$ (or airmass) at the alert's
$(\alpha, \delta, \mathrm{MJD})$.

### 6a. Select the alert

In [ ]:
idx_frame = 0
idx_alert = 0

### 6b. Display alert row and extract coordinates

In [ ]:
one_alert_row = frames[idx_frame].iloc[idx_alert]
one_alert_row

In [ ]:
field_name_al = one_alert_row["field"]
mjd_al = one_alert_row["r:midpointMjdTai"]
t_al = Time(mjd_al, format="mjd", scale="tai")
t_al_datetime = t_al.utc.datetime
ra_al = one_alert_row["r:ra"]
dec_al = one_alert_row["r:dec"]
print(ra_al, dec_al, t_al.iso, mjd_al, field_name_al)

### 6c. Extract geometry scalars and define astroplan `FixedTarget`

We convert the alert's MJD to a UTC `datetime` (required by some astroplan
plotting functions) and build the `FixedTarget` object.

In [ ]:
para_angle_al = one_alert_row["parallactic_angle_deg"]
para_angle_al_rad = np.radians(para_angle_al)
airmass_al = one_alert_row["airmass"]
azimuth_al = one_alert_row["azimuth_deg"]
zenith_al = one_alert_row["zenith_angle_deg"]
sinzenith_al = one_alert_row["sin_zenith"]
altitude_al = one_alert_row["altitude_deg"]
hour_angle = one_alert_row["hour_angle_deg"]

In [ ]:
coords = SkyCoord(ra_al * u.deg, dec_al * u.deg, frame="icrs")
field_target = FixedTarget(name=field_name_al, coord=coords)

### 6d. Astroplan `plot_parallactic` — nightly η curve

The red dot marks the value of η computed by `compute_observing_geometry`
at the exact alert time.  It should fall on the astroplan curve.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_parallactic(
    field_target,
    observer,
    t_al,
    ax=ax,
)
ax.scatter([t_al_datetime], [para_angle_al_rad], marker="o", s=20, color="red")


ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d:%H:%M"))
ax.xaxis.set_minor_locator(mdates.MinuteLocator(interval=15))
fig.autofmt_xdate()


ax.set_title("Astroplan parallactic angle calculation")
plt.show()

### 6e. My model η(H) — all DDFs with alert overlay

The alert's $(H, \eta)$ pair (blue dot) is overlaid on the analytic η(H)
curves.  The dot should lie exactly on the curve of the alert's field.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")

HA = np.arange(-180.0, 180.0) * u.deg

for key, value in DEEP_FIELDS.items():
    field_name = key
    ra = value[0]
    dec = value[1]
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    label = field_name + " : ( $\delta = $" + f"{dec:.1f} deg )"

    q = calculate_parallactic_angle_fromHA(HA, coords, RUBIN_LOCATION)

    ax.plot(HA, q, label=label, lw=1)


ax.scatter([hour_angle], [para_angle_al], marker="o", s=20, color="b", label="alert")

ax.set_xlabel("Hour angle (degrees)")
ax.set_ylabel("Parallactic angle (degrees)")


# --- Axe secondaire en heures ---
def deg2hour(x):
    return x / 15.0


def hour2deg(x):
    return x * 15.0


secax = ax.secondary_xaxis("top", functions=(deg2hour, hour2deg))
secax.set_xlabel("Hour angle (hours)")


ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.01, 1))

ax.set_title(f"LSST Deep Fields parallactic angle vs hour angle")

plt.show()

### 6f. My model $\sin z(H)$ — all DDFs with alert overlay

Same cross-check for $\sin z$.  The blue dot (alert) should lie on the
curve of its DDF.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")

HA = np.arange(-180.0, 180.0) * u.deg

for key, value in DEEP_FIELDS.items():
    field_name = key
    ra = value[0]
    dec = value[1]
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    label = field_name + " : ( $\delta = $" + f"{dec:.1f} deg )"

    # projection of zenith in object tangeant plane
    sinz = sinz_vs_HA(HA, coords, RUBIN_LOCATION)

    ax.plot(HA, sinz, label=label, lw=0.5)

ax.set_xlabel("Hour angle (degrees)")
ax.set_ylabel("sinz")


# --- Axe secondaire en heures ---
def deg2hour(x):
    return x / 15.0


def hour2deg(x):
    return x * 15.0


ax.scatter([hour_angle], [sinzenith_al], marker="o", s=20, color="blue", label="alert")

secax = ax.secondary_xaxis("top", functions=(deg2hour, hour2deg))
secax.set_xlabel("Hour angle (hours)")

ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"LSST Deep Fields sinz vs hour angle")

plt.show()

### 6g. Astroplan `plot_airmass` — nightly airmass + altitude curve

The red dot marks the airmass value computed by `compute_observing_geometry`.
Shaded regions indicate twilight / daytime.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_airmass(
    field_target,
    observer,
    t_al_datetime,
    ax=ax,
    brightness_shading=True,  # nuit/jour
    altitude_yaxis=True,  # altitude en plus
)
ax.scatter([t_al_datetime], [airmass_al], marker="o", s=20, color="red")


ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d:%H:%M"))
ax.xaxis.set_minor_locator(mdates.MinuteLocator(interval=15))
fig.autofmt_xdate()


ax.set_title("Astroplan parallactic angle calculation")
plt.show()

### 6h. Astroplan `plot_sky` — polar alt-az chart

The open circle (blue) marks the alert's position in (azimuth, zenith)
polar coordinates.  North is up, East is to the right (θ direction is
reversed so East appears on the right as in a standard sky chart).

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4), subplot_kw={"projection": "polar"}, layout="constrained")

# east on the right, west on the left : -1
ax.set_theta_direction(-1)

plot_sky(
    field_target,
    observer,
    t_al,  # ⚠️ ici tu peux garder Time, pas besoin de datetime
    ax=ax,
    style_kwargs=DEEP_FIELDS_COLORSTYLE[field_name_al],
)

# --- point à ajouter ---
az = azimuth_al  # degrés
alt = altitude_al
z = zenith_al

ax.scatter(
    np.deg2rad(az),
    z,
    facecolors="none",  # pas de remplissage
    edgecolors="blue",  # couleur du contour
    s=60,
    marker="o",
    label="alert",
)


ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.02, 1))
ax.set_title(f"LSST Deep Fields for day {t_al.iso}")

plt.show()

## 7. Summary

The three independent calculations agree on η, $\sin z$, and airmass
for the selected alert:

| Quantity | `compute_observing_geometry` | My analytic model | astroplan |
|----------|-----------------------------:|------------------:|----------:|
| η (deg)  | `parallactic_angle_deg`      | `calculate_parallactic_angle_fromHA` | `plot_parallactic` |
| sin z    | `sin_zenith`                 | `sinz_vs_HA`       | derived from `plot_airmass` |
| airmass  | `airmass`                    | 1/cos z            | `plot_airmass` |

This validates both the angle-convention pipeline and the `compute_observing_geometry`
function used on the full alert catalogue in subsequent analysis notebooks
(e.g. `05b_...`, `07_...`).
